# Imports

In [ ]:
import os, json, time, copy
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import defaultdict
import glob

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
import faiss
import kagglehub

tushar5harma_path    = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
shubham2703_path     = kagglehub.dataset_download('shubham2703/five-crop-diseases-dataset')
zunorain_path        = kagglehub.dataset_download('zunorain/pea-plant-dataset')
mexwell_path         = kagglehub.dataset_download('mexwell/crop-diseases-classification')
ayushbpanchal_path   = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')
tahmidmir_path       = kagglehub.dataset_download('tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh')
binnassor89_path     = kagglehub.dataset_download('binnassor89/melon-macronutrient-deficiency-dataset')
nirmalsankalana_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')
yasaminborhani_path = kagglehub.dataset_download('yasaminborhani/wheatrustclassificationdataset')

print('Downloads complete.')

# Config

In [ ]:
runs_dir = 'kaggle/working/runs'
os.makedirs(runs_dir, exist_ok=True)

base = 'kaggle/input/datasets'
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

batch_size = 64
num_workers = 4
seed = 42
n_epochs = 20 
img_size = 224
early_stop_patience = 4

lr_head   = 5e-4 
lr_top    = 5e-5
lr_mid    = 2e-5
lr_bottom = 1e-5

torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

# Label Maps & Class Index

In [ ]:
dataset_label_maps = {

    'tushar5harma': {
        ('Apple',        'Apple Scab')             : 'Apple__Apple_Scab',
        ('Apple',        'Black Rot')              : 'Apple__Black_Rot',
        ('Apple',        'Cedar Apple Rust')       : 'Apple__Cedar_Apple_Rust',
        ('Apple',        'Healthy')                : 'Apple__Healthy',
        ('Bell Pepper',  'Bacterial Spot')         : 'Bell_Pepper__Bacterial_Spot',
        ('Bell Pepper',  'Healthy')                : 'Bell_Pepper__Healthy',
        ('Cherry',       'Healthy')                : 'Cherry__Healthy',
        ('Cherry',       'Powdery Mildew')         : 'Cherry__Powdery_Mildew',
        ('Corn (Maize)', 'Cercospora Leaf Spot')   : 'Corn__Gray_Leaf_Spot',
        ('Corn (Maize)', 'Common Rust')            : 'Corn__Common_Rust',
        ('Corn (Maize)', 'Healthy')                : 'Corn__Healthy',
        ('Corn (Maize)', 'Northern Leaf Blight')   : 'Corn__Northern_Leaf_Blight',
        ('Grape',        'Black Rot')              : 'Grape__Black_Rot',
        ('Grape',        'Esca (Black Measles)')   : 'Grape__Black_Measles',
        ('Grape',        'Healthy')                : 'Grape__Healthy',
        ('Grape',        'Leaf Blight')            : 'Grape__Leaf_Blight',
        ('Peach',        'Bacterial Spot')         : 'Peach__Bacterial_Spot',
        ('Peach',        'Healthy')                : 'Peach__Healthy',
        ('Potato',       'Early Blight')           : 'Potato__Early_Blight',
        ('Potato',       'Healthy')                : 'Potato__Healthy',
        ('Potato',       'Late Blight')            : 'Potato__Late_Blight',
        ('Strawberry',   'Healthy')                : 'Strawberry__Healthy',
        ('Strawberry',   'Leaf Scorch')            : 'Strawberry__Leaf_Scorch',
        ('Tomato',       'Bacterial Spot')         : 'Tomato__Bacterial_Spot',
        ('Tomato',       'Early Blight')           : 'Tomato__Early_Blight',
        ('Tomato',       'Healthy')                : 'Tomato__Healthy',
        ('Tomato',       'Late Blight')            : 'Tomato__Late_Blight',
        ('Tomato',       'Septoria Leaf Spot')     : 'Tomato__Septoria_Leaf_Spot',
        ('Tomato',       'Yellow Leaf Curl Virus') : 'Tomato__Yellow_Leaf_Curl_Virus',
    },

    'shubham2703': {
        'Corn___Common_Rust'          : 'Corn__Common_Rust',
        'Corn___Gray_Leaf_Spot'       : 'Corn__Gray_Leaf_Spot',
        'Corn___Healthy'              : 'Corn__Healthy',
        'Corn___Northern_Leaf_Blight' : 'Corn__Northern_Leaf_Blight',
        'Potato___Early_Blight'       : 'Potato__Early_Blight',
        'Potato___Healthy'            : 'Potato__Healthy',
        'Potato___Late_Blight'        : 'Potato__Late_Blight',
        'Rice___Brown_Spot'           : 'Rice__Brown_Spot',
        'Rice___Healthy'              : 'Rice__Healthy',
        'Rice___Leaf_Blast'           : 'Rice__Leaf_Blast',
        'Rice___Neck_Blast'           : 'Rice__Neck_Blast',
        'Wheat___Brown_Rust'          : 'Wheat__Brown_Rust',
        'Wheat___Healthy'             : 'Wheat__Healthy',
        'Wheat___Yellow_Rust'         : 'Wheat__Yellow_Rust',
        'Bacterial Blight'            : 'Rice__Bacterial_Blight',
        'Red Rot'                     : 'Sugarcane__Red_Rot',
    },

    'nirmalsankalana': {
        'coffee___healthy'         : 'Coffee__Healthy',
        'coffee___red_spider_mite' : 'Coffee__Red_Spider_Mite',
        'coffee___rust'            : 'Coffee__Rust',
        'BlackMeasles'             : 'Grape__Black_Measles',
        'BlackRot'                 : 'Grape__Black_Rot',
        'HealthyGrapes'            : 'Grape__Healthy',
        'LeafBlight'               : 'Grape__Leaf_Blight',
    },

    'binnassor89': {
        'calcium'   : 'Melon__Calcium_Deficiency',
        'health'    : 'Melon__Healthy',
        'nitrogen'  : 'Melon__Nitrogen_Deficiency',
        'potasium'  : 'Melon__Potassium_Deficiency',
    },

    'zunorain': {
        'DOWNY_MILDEW_LEAF'  : 'Pea__Downy_Mildew',
        'FRESH_LEAF'         : 'Pea__Healthy',
        'LEAFMINNER_LEAF'    : 'Pea__Leaf_Miner',
        'POWDER_MILDEW_LEAF' : 'Pea__Powdery_Mildew',
    },

    'ayushbpanchal': {
        'anthracnose' : 'Cashew__Anthracnose',
        'gumosis'     : 'Cashew__Gummosis',
        'healthy'     : 'Cashew__Healthy',
        'leaf miner'  : 'Cashew__Leaf_Miner',
        'red rust'    : 'Cashew__Red_Rust',
    },

    'tahmidmir': {
        'Bacterial Leaf Spot' : 'Pumpkin__Bacterial_Leaf_Spot',
        'Downy Mildew'        : 'Pumpkin__Downy_Mildew',
        'Healthy Leaf'        : 'Pumpkin__Healthy',
        'Mosaic Disease'      : 'Pumpkin__Mosaic_Disease',
        'Powdery_Mildew'      : 'Pumpkin__Powdery_Mildew',
    },

    'mexwell': {
        'Cassava Bacterial Blight (CBB)'      : 'Cassava__Bacterial_Blight',
        'Cassava Brown Streak Disease (CBSD)' : 'Cassava__Brown_Streak_Disease',
        'Cassava Green Mottle (CGM)'          : 'Cassava__Green_Mottle',
        'Cassava Mosaic Disease (CMD)'        : 'Cassava__Mosaic_Disease',
        'Healthy'                             : 'Cassava__Healthy',
    },

    'yasaminborhani': {
        'Brown_rust'  : 'Wheat__Brown_Rust',
        'Healthy'     : 'Wheat__Healthy',
        'Yellow_rust' : 'Wheat__Yellow_Rust',
    },
}

all_cannonical = sorted(set(
    v for mapping in dataset_label_maps.values()
    for v in mapping.values()
    if v is not None
))

class_to_idx = {cls: idx for idx, cls in enumerate(all_cannonical)}
idx_to_class = {idx: cls for cls, idx in enumerate(all_cannonical)}
n_classes = len(class_to_idx)
print(f"Total cannonical classes: {n_classes}")

# Sample Builders 

In [ ]:
def build_samples_folder(slug):
    label_map = dataset_label_maps[slug]
    root = Path(base) / slug
    samples, skipped = [], 0
    for img_path in root.rglob('*'):
        if img_path.suffix.lower() not in image_exts:
            continue
        cannonical = label_map.get(img_path.parent.name)
        if cannonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[cannonical]))
    print(f"[{slug:<20}] {len(samples):>6} samples      ({skipped} skipped)")
    return samples

def build_samples_tushar():
    label_map = dataset_label_maps['tushar5harma']
    root      = Path(base) / 'tushar5harma'
    samples, skipped = [], 0
    for img_path in root.rglob('*'):
        if img_path.suffix.lower() not in image_exts:
            continue
        disease_dir = img_path.parent
        split_dir   = disease_dir.parent
        crop_dir    = split_dir.parent
        key = (crop_dir.name, disease_dir.name)
        canonical = label_map.get(key)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f'  [{"tushar5harma":<20}] {len(samples):>6} samples  ({skipped} skipped)')
    return samples


def build_samples_mexwell():
    data_root = Path(base) / 'mexwell' / 'crop-diseases-classification' / 'Data'
    img_dir   = data_root / 'train_images'
    json_path = data_root / 'label_num_to_disease_map.json'
    csv_path  = data_root / 'train.csv'
    with open(json_path) as f:
        num_to_name = json.load(f)
    df        = pd.read_csv(csv_path)
    label_map = dataset_label_maps['mexwell']
    samples, skipped = [], 0
    for _, row in df.iterrows():
        img_path     = img_dir / row['image_id']
        if not img_path.exists():
            skipped += 1
            continue
        disease_name = num_to_name[str(row['label'])]
        canonical    = label_map.get(disease_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f'  [{"mexwell":<20}] {len(samples):>6} samples  ({skipped} skipped)')
    return samples


def build_all_samples():
    print('Loading datasets...')
    folder_slugs = ['shubham2703', 'nirmalsankalana', 'binnassor89',
                    'zunorain', 'ayushbpanchal', 'tahmidmir', 'yasaminborhani']
    all_samples = []
    for slug in folder_slugs:
        all_samples += build_samples_folder(slug)
    all_samples += build_samples_mexwell()
    all_samples += build_samples_tushar()
    print(f'\nTotal: {len(all_samples):,} samples | '
          f'{len(set(lbl for _, lbl in all_samples))} unique class indices')
    return all_samples

# Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class CropDiseaseDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# DataLoaders

In [ ]:
def build_dataloaders(all_samples):
    labels = [lbl for _, lbl in all_samples]
    train_s, val_s = train_test_split(
        all_samples, test_size=0.15, random_state=seed, stratify=labels
    )
    train_ds = CropDiseaseDataset(train_s, transform=train_transform)
    val_ds   = CropDiseaseDataset(val_s,   transform=val_transform)

    train_labels  = [lbl for _, lbl in train_s]
    class_counts  = np.bincount(train_labels, minlength=n_classes)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_w      = [class_weights[l] for l in train_labels]
    sampler       = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    kw = dict(num_workers=num_workers, prefetch_factor=4,
              persistent_workers=True, pin_memory=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,   **kw)
    print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')
    return train_loader, val_loader, train_s, val_s

# Backbone

In [ ]:
class ResNet50Backbone(nn.Module):
    feature_dim = 2048

    def __init__(self, n_classes, dropout = 0.4):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.head    = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )
    
    def extract_features(self, x):
        return self.encoder(x).flatten(1)
    
    def forward(self, x):
        return self.head(self.extract_features(x))
    

class EfficientNetB3Backbone(nn.Module):
    feature_dim = 1536

    def __init__(self, n_classes, dropout = 0.4):
        super().__init__()
        base = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.encoder  = base.features 
        self.pool     = base.avgpool
        self.head    = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )

    def extract_features(self, x):
        return self.pool(self.encoder(x)).flatten(1)
    
    def forward(self, x):
        return self.head(self.extract_features(x))
    

class DenseNet121Backbone(nn.Module):
    feature_dim = 1024

    def __init__(self, n_classes, dropout = 0.4):
        super().__init__()
        base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.encoder = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head    = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )

    def extract_features(self, x):
        feat = self.pool(F.relu(self.encoder(x), inplace=True))
        return feat.flatten(1)
    
    def forward(self, x):
        return self.head(self.extract_features(x))
    

def build_all_backbones(n_classes):
    backbones = {
        'resnet50'  : ResNet50Backbone(n_classes).to(device),
        'efficient' : EfficientNetB3Backbone(n_classes).to(device),
        'densenet'  : DenseNet121Backbone(n_classes).to(device),
    }
    for name, m in backbones.items():
        n_params = sum(p.numel() for p in m.parameters()) / 1e6
        print(f"{name:<15} {n_params:.1f}M params")
    return backbones

# Optimizers

In [ ]:
def build_optimizer_resnet(model):
    base = model.module if hasattr(model, 'module') else model
    enc  = list(base.encoder.children())
    return optim.AdamW([
        {'params': list(enc[0].parameters()) + list(enc[1].parameters()),
         'lr': lr_bottom, 'name': 'conv1_bn1'},
        {'params': enc[4].parameters(), 'lr': lr_bottom, 'name': 'layer1'},
        {'params': enc[5].parameters(), 'lr': lr_mid,    'name': 'layer2'},
        {'params': enc[6].parameters(), 'lr': lr_top,    'name': 'layer3'},
        {'params': enc[7].parameters(), 'lr': lr_top,    'name': 'layer4'},
        {'params': base.head.parameters(), 'lr': lr_head, 'name': 'head'},
    ], weight_decay=1e-4)


def build_optimizer_efficientnet(model):
    base = model.module if hasattr(model, 'module') else model
    enc  = list(base.encoder.children())
    n    = len(enc)
    return optim.AdamW([
        {'params': enc[0].parameters(), 'lr': lr_bottom, 'name': 'block0'},
        {'params': sum([list(enc[i].parameters()) for i in range(1, n//2)], []),
         'lr': lr_mid, 'name': 'blocks_mid'},
        {'params': sum([list(enc[i].parameters()) for i in range(n//2, n)], []),
         'lr': lr_top, 'name': 'blocks_top'},
        {'params': base.head.parameters(), 'lr': lr_head, 'name': 'head'},
    ], weight_decay=1e-4)


def build_optimizer_densenet(model):
    base = model.module if hasattr(model, 'module') else model
    enc  = list(base.encoder.children())
    n    = len(enc)
    return optim.AdamW([
        {'params': enc[0].parameters(), 'lr': lr_bottom, 'name': 'conv0'},
        {'params': sum([list(enc[i].parameters()) for i in range(1, n//2)], []),
         'lr': lr_mid, 'name': 'blocks_early'},
        {'params': sum([list(enc[i].parameters()) for i in range(n//2, n)], []),
         'lr': lr_top, 'name': 'blocks_late'},
        {'params': base.head.parameters(), 'lr': lr_head, 'name': 'head'},
    ], weight_decay=1e-4)


def build_optimizers(backbones):
    return {
        'resnet50'    : build_optimizer_resnet(backbones['resnet50']),
        'efficientnet': build_optimizer_efficientnet(backbones['efficientnet']),
        'densenet'    : build_optimizer_densenet(backbones['densenet']),
    }

# Training Utilities

In [ ]:
def accuracy_topk(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        bs   = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        correct = pred.t().eq(target.view(1, -1).expand_as(pred.t()))
        return [correct[:k].reshape(-1).float().sum().mul_(100.0 / bs).item() for k in topk]


def mixup_data(x, y, alpha=0.4):
    lam     = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx     = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(name, model, loader, criterion, optimizer, epoch, use_mixup=True):
    model.train()
    loss_sum = top1_sum = top5_sum = 0.0
    t0 = time.time()
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        if use_mixup:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels)
        optimizer.zero_grad()
        out = model(imgs)
        loss = mixup_criterion(criterion, out, y_a, y_b, lam) if use_mixup else criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        top1, top5 = accuracy_topk(out, labels)
        loss_sum += loss.item(); top1_sum += top1; top5_sum += top5
        if (i + 1) % 100 == 0:
            print(f'  [{name}|E{epoch} {i+1}/{len(loader)}] '
                  f'loss={loss_sum/(i+1):.4f}  top1={top1_sum/(i+1):.2f}%  '
                  f't={time.time()-t0:.0f}s')
    n = len(loader)
    return loss_sum / n, top1_sum / n, top5_sum / n


@torch.no_grad()
def evaluate(name, model, loader):
    model.eval()
    t1 = t5 = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        a1, a5 = accuracy_topk(out, labels)
        bs = labels.size(0)
        t1 += a1 * bs; t5 += a5 * bs; total += bs
    top1, top5 = t1 / total, t5 / total
    print(f'  [{name}] Val top1={top1:.2f}%  top5={top5:.2f}%')
    return top1, top5

# Checkpoint Save / Load

In [ ]:
def save_checkpoint(name, model, optimizer, scheduler, epoch, val_top1, tag, history=None):
    base = model.module if hasattr(model, 'module') else model
    path = os.path.join(runs_dir, f'{name}_{tag}.pth')
    torch.save({
        'backbone':             name,
        'epoch':                epoch,
        'model_state_dict':     base.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_acc':             val_top1,
        'class_to_idx':         class_to_idx,
        'n_classes':            n_classes,
        'history':              history or [],
    }, path)
    print(f'  → {name}_{tag}.pth  (top1={val_top1:.4f})')
    return path


def load_checkpoint(name, model, path):
    ckpt  = torch.load(path, map_location='cpu')
    state = ckpt['model_state_dict']
    if any(k.startswith('module.') for k in state):
        state = {k.replace('module.', '', 1): v for k, v in state.items()}
    base  = model.module if hasattr(model, 'module') else model
    base.load_state_dict(state)
    print(f'  Loaded {name} | epoch={ckpt["epoch"]} | best_acc={ckpt["best_acc"]:.4f}')
    return ckpt

# Training Loop

In [ ]:
def train_backbone(name, model, train_loader, val_loader, optimizer, n_epochs=n_epochs):
    resume_path = os.path.join(runs_dir, f'{name}_latest.pth')
    scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-7)
    criterion   = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_epoch = 0
    best_top1   = 0.0
    no_improve  = 0
    history     = []

    if os.path.exists(resume_path):
        print(f'\nResuming {name} from {resume_path}')
        ckpt  = torch.load(resume_path, map_location='cpu')
        state = ckpt['model_state_dict']
        if any(k.startswith('module.') for k in state):
            state = {k.replace('module.', '', 1): v for k, v in state.items()}
        base  = model.module if hasattr(model, 'module') else model
        base.load_state_dict(state)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_top1   = ckpt.get('best_acc', 0.0)
        history     = ckpt.get('history', [])
        print(f'  Resumed at epoch {start_epoch}, best_top1={best_top1:.4f}')

    print(f'\n{"="*60}')
    print(f'Training: {name}  |  epochs {start_epoch}..{n_epochs-1}')
    print(f'{"="*60}')

    for epoch in range(start_epoch, n_epochs):
        tr_loss, tr1, tr5 = train_one_epoch(name, model, train_loader, criterion, optimizer, epoch)
        val_top1, val_top5 = evaluate(name, model, val_loader)
        scheduler.step()

        history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_top1=tr1,
                            val_top1=val_top1, val_top5=val_top5))
        print(f'  Epoch {epoch}: train_top1={tr1:.2f}%  val_top1={val_top1:.2f}%')

        save_checkpoint(name, model, optimizer, scheduler, epoch, val_top1, 'latest', history)

        if val_top1 > best_top1:
            best_top1  = val_top1
            no_improve = 0
            save_checkpoint(name, model, optimizer, scheduler, epoch, val_top1, 'best', history)
        else:
            no_improve += 1
            print(f'  No improvement ({no_improve}/{early_stop_patience})')
            if no_improve >= early_stop_patience:
                print(f'  Early stopping {name} at epoch {epoch}')
                break

    print(f'\n{name} done. Best val top1: {best_top1:.4f}')
    return os.path.join(runs_dir, f'{name}_best.pth')

In [ ]:
all_samples = build_all_samples()
train_loader, val_loader, train_s, val_s = build_dataloaders(all_samples)

In [ ]:
backbones  = build_all_backbones(n_classes)
optimizers = build_optimizers(backbones)

best_ckpt_paths = {}
for name in ['resnet50', 'efficientnet', 'densenet']:
    best_ckpt_paths[name] = train_backbone(
        name, backbones[name], train_loader, val_loader, optimizers[name]
    )

print('\nAll backbones trained.')
print(best_ckpt_paths)

# Ensemble Evaluation

In [ ]:
def load_trained_backbones(ckpt_paths):
    loaded = {}
    cls_map = {'resnet50': ResNet50Backbone,
               'efficientnet': EfficientNetB3Backbone,
               'densenet': DenseNet121Backbone}
    for name, path in ckpt_paths.items():
        m = cls_map[name](n_classes)
        load_checkpoint(name, m, path)
        loaded[name] = m.to(device).eval()
    return loaded


@torch.no_grad()
def ensemble_predict(models_dict, imgs):
    probs_list = [F.softmax(m(imgs), dim=1) for m in models_dict.values()]
    return torch.stack(probs_list, dim=0).mean(0)


@torch.no_grad()
def evaluate_ensemble(models_dict, loader):
    top1_sum = top5_sum = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        probs = ensemble_predict(models_dict, imgs)
        a1, a5 = accuracy_topk(probs, labels)
        bs = labels.size(0)
        top1_sum += a1 * bs; top5_sum += a5 * bs; total += bs
    print(f'  Ensemble Val top1={top1_sum/total:.2f}%  top5={top5_sum/total:.2f}%')
    return top1_sum / total, top5_sum / total


trained_models = load_trained_backbones(best_ckpt_paths)
evaluate_ensemble(trained_models, val_loader)

# Feature Extraction & FAISS

In [ ]:
concat_dim = (ResNet50Backbone.feature_dim +
              EfficientNetB3Backbone.feature_dim +
              DenseNet121Backbone.feature_dim)   # 4608

bank_prefix = '/kaggle/working/runs/feature_bank'


@torch.no_grad()
def extract_all_features(models_dict, samples, batch_size=256):
    for m in models_dict.values():
        m.eval()
    ds     = CropDiseaseDataset(samples, transform=val_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    all_feats, all_labels = [], []
    for i, (imgs, lbls) in enumerate(loader):
        imgs  = imgs.to(device)
        parts = [m.extract_features(imgs) for m in models_dict.values()]
        concat = F.normalize(torch.cat(parts, dim=1), p=2, dim=1)
        all_feats.append(concat.cpu().numpy())
        all_labels.extend(lbls.numpy())
        if (i + 1) % 50 == 0:
            print(f'  Extracted {(i+1)*batch_size:,} / {len(ds):,}')
    features = np.vstack(all_feats).astype(np.float32)
    labels   = np.array(all_labels, dtype=np.int64)
    print(f'Features: {features.shape}')
    return features, labels


def build_faiss_index(features):
    index = faiss.IndexFlatL2(features.shape[1])
    index.add(features)
    print(f'FAISS index: {index.ntotal:,} vectors  dim={features.shape[1]}')
    return index


def save_feature_bank(features, labels, index, prefix):
    np.save(f'{prefix}_features.npy', features)
    np.save(f'{prefix}_labels.npy',   labels)
    faiss.write_index(index, f'{prefix}_faiss.index')
    print(f'Saved to {prefix}_*')


def load_feature_bank(prefix):
    features = np.load(f'{prefix}_features.npy')
    labels   = np.load(f'{prefix}_labels.npy')
    index    = faiss.read_index(f'{prefix}_faiss.index')
    print(f'Loaded bank: {index.ntotal:,} vectors')
    return features, labels, index


print('Extracting features from training set...')
bank_features, bank_labels = extract_all_features(trained_models, train_s)
faiss_index = build_faiss_index(bank_features)
save_feature_bank(bank_features, bank_labels, faiss_index, bank_prefix)

# CropQuery Predictor

In [ ]:
class CropQueryPredictor:
    def __init__(self, models_dict, features, labels, index,
                 class_to_idx, idx_to_class, k=7):
        self.models       = models_dict
        self.features     = features
        self.labels       = labels
        self.index        = index
        self.class_to_idx = dict(class_to_idx)
        self.idx_to_class = dict(idx_to_class)
        self.k            = k
        self._next_idx    = max(idx_to_class.keys()) + 1

    def _get_features(self, image_paths):
        feats = []
        for path in image_paths:
            img = val_transform(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
            with torch.no_grad():
                parts = [m.extract_features(img) for m in self.models.values()]
            concat = F.normalize(torch.cat(parts, dim=1), p=2, dim=1)
            feats.append(concat.cpu().numpy())
        return np.vstack(feats).astype(np.float32)

    def predict_ensemble(self, image_path):
        img   = val_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = ensemble_predict(self.models, img)[0]
        idx = probs.argmax().item()
        return self.idx_to_class[idx], probs[idx].item()

    def predict_knn(self, image_path):
        feat = self._get_features([image_path])
        _, I = self.index.search(feat, self.k)
        nb_labels = self.labels[I[0]]
        counts    = np.bincount(nb_labels, minlength=len(self.idx_to_class))
        best_idx  = counts.argmax()
        return self.idx_to_class[best_idx], counts[best_idx] / self.k

    def add_new_class(self, class_name, support_image_paths):
        if class_name in self.class_to_idx:
            new_idx = self.class_to_idx[class_name]
            print(f'{class_name} exists — appending support images.')
        else:
            new_idx = self._next_idx
            self.class_to_idx[class_name] = new_idx
            self.idx_to_class[new_idx]    = class_name
            self._next_idx += 1
            print(f'New class: {class_name} → idx {new_idx}')

        new_feats  = self._get_features(support_image_paths)
        new_labels = np.full(len(support_image_paths), new_idx, dtype=np.int64)
        self.features = np.vstack([self.features, new_feats])
        self.labels   = np.concatenate([self.labels, new_labels])
        self.index.add(new_feats)
        print(f'  Bank size: {self.index.ntotal:,}')
        return new_idx

    def save_updated_bank(self, prefix):
        save_feature_bank(self.features, self.labels, self.index, prefix)
        with open(f'{prefix}_class_map.json', 'w') as f:
            json.dump({'class_to_idx': self.class_to_idx,
                       'idx_to_class': {str(k): v for k, v in self.idx_to_class.items()}}, f, indent=2)


predictor = CropQueryPredictor(
    models_dict  = trained_models,
    features     = bank_features,
    labels       = bank_labels,
    index        = faiss_index,
    class_to_idx = class_to_idx,
    idx_to_class = idx_to_class,
    k            = 7,
)
print('Predictor ready.')

# k-NN Accuracy on Val Set

In [ ]:
@torch.no_grad()
def knn_accuracy_on_val(predictor, val_samples, batch_size=256):
    ds     = CropDiseaseDataset(val_samples, transform=val_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    correct = total = 0
    for imgs, labels in loader:
        imgs  = imgs.to(device)
        parts = [m.extract_features(imgs) for m in predictor.models.values()]
        feats = F.normalize(torch.cat(parts, dim=1), p=2, dim=1).cpu().numpy()
        _, I  = predictor.index.search(feats, predictor.k)
        for neighs, true_lbl in zip(I, labels.numpy()):
            pred     = np.bincount(predictor.labels[neighs]).argmax()
            correct += int(pred == true_lbl)
            total   += 1
    acc = 100 * correct / total
    print(f'k-NN (k={predictor.k}) Val Accuracy: {acc:.2f}%  ({correct}/{total})')
    return acc

knn_acc = knn_accuracy_on_val(predictor, val_s)